# Lab 03 - Divided and Finite Differences

Solutions for all tasks from `L3_NC.pdf`, implemented in Python.

This notebook contains:
1. A function for divided differences with simple nodes
2. A function for divided differences with double nodes using derivative values
3. Functions for forward and backward finite differences
4. The required applications from the statement

In [1]:
import html

import numpy as np
from IPython.display import HTML, display


def _format_value(value: float, precision: int = 8) -> str:
    if np.isnan(value):
        return "0"
    return f"{value:.{precision}g}"


def display_difference_table(title: str, node_label: str, nodes: np.ndarray, table: np.ndarray, precision: int = 8) -> None:
    headers = [node_label] + [f"order {k}" for k in range(table.shape[1])]
    header_html = "".join(f"<th>{html.escape(header)}</th>" for header in headers)

    rows_html = []
    for node, row in zip(nodes, table):
        cells = [f"<td>{_format_value(node, precision)}</td>"]
        cells.extend(f"<td>{_format_value(value, precision)}</td>" for value in row)
        rows_html.append("<tr>" + "".join(cells) + "</tr>")

    table_html = f"""
    <h4>{html.escape(title)}</h4>
    <table style='border-collapse: collapse; font-family: monospace;'>
        <thead>
            <tr>{header_html}</tr>
        </thead>
        <tbody>
            {''.join(rows_html)}
        </tbody>
    </table>
    """
    display(HTML(table_html))


def divided_differences_simple(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if x.ndim != 1 or y.ndim != 1 or len(x) != len(y):
        raise ValueError("x and y must be one-dimensional arrays of the same length")

    n = len(x)
    table = np.full((n, n), np.nan, dtype=float)
    table[:, 0] = y

    for order in range(1, n):
        for i in range(order, n):
            numerator = table[i, order - 1] - table[i - 1, order - 1]
            denominator = x[i] - x[i - order]
            table[i, order] = numerator / denominator

    return x, table


def divided_differences_double(x: np.ndarray, y: np.ndarray, dy: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    dy = np.asarray(dy, dtype=float)

    if x.ndim != 1 or y.ndim != 1 or dy.ndim != 1:
        raise ValueError("x, y, and dy must be one-dimensional arrays")
    if not (len(x) == len(y) == len(dy)):
        raise ValueError("x, y, and dy must have the same length")

    n = len(x)
    m = 2 * n
    z = np.repeat(x, 2)
    table = np.full((m, m), np.nan, dtype=float)

    for i in range(n):
        left = 2 * i
        right = left + 1
        table[left, 0] = y[i]
        table[right, 0] = y[i]
        table[right, 1] = dy[i]

        if i > 0:
            table[left, 1] = (table[left, 0] - table[left - 1, 0]) / (z[left] - z[left - 1])

    for order in range(2, m):
        for i in range(order, m):
            numerator = table[i, order - 1] - table[i - 1, order - 1]
            denominator = z[i] - z[i - order]
            table[i, order] = numerator / denominator

    return z, table


def forward_differences(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    n = len(y)
    table = np.full((n, n), np.nan, dtype=float)
    table[:, 0] = y

    for order in range(1, n):
        for i in range(n - order):
            table[i, order] = table[i + 1, order - 1] - table[i, order - 1]

    return table


def backward_differences(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    n = len(y)
    table = np.full((n, n), np.nan, dtype=float)
    table[:, 0] = y

    for order in range(1, n):
        for i in range(order, n):
            table[i, order] = table[i, order - 1] - table[i - 1, order - 1]

    return table

## Exercise 1

Let

$$
f(x) = \frac{1}{1+x}, \qquad f'(x) = -\frac{1}{(1+x)^2}.
$$

We compute:
- the divided differences table at the simple nodes $x_0 = 0$, $x_1 = 1$, $x_2 = 2$
- the divided differences table at the corresponding double nodes
- the same two tables for 11 equidistant nodes on $[1, 2]$

In [2]:
f = lambda x: 1.0 / (1.0 + x)
df = lambda x: -1.0 / (1.0 + x) ** 2

# 1a) Simple nodes x = [0, 1, 2]
x_small = np.array([0.0, 1.0, 2.0])
y_small = f(x_small)
nodes_simple_small, table_simple_small = divided_differences_simple(x_small, y_small)
display_difference_table("Exercise 1a - divided differences (simple nodes)", "x", nodes_simple_small, table_simple_small)
print("Newton coefficients for 1a:", np.diag(table_simple_small))

# 1b) Double nodes x = [0, 1, 2]
z_small, table_double_small = divided_differences_double(x_small, y_small, df(x_small))
display_difference_table("Exercise 1b - divided differences (double nodes)", "z", z_small, table_double_small)
print("Hermite/Newton coefficients for 1b:", np.diag(table_double_small))

# 1c) 11 equidistant nodes on [1, 2]
x_large = np.linspace(1.0, 2.0, 11)
y_large = f(x_large)

nodes_simple_large, table_simple_large = divided_differences_simple(x_large, y_large)
display_difference_table("Exercise 1c - divided differences at 11 simple nodes on [1, 2]", "x", nodes_simple_large, table_simple_large, precision=10)
print("Newton coefficients for 1c (simple nodes):", np.diag(table_simple_large))

z_large, table_double_large = divided_differences_double(x_large, y_large, df(x_large))
display_difference_table("Exercise 1c - divided differences at 11 double nodes on [1, 2]", "z", z_large, table_double_large, precision=10)
print("Hermite/Newton coefficients for 1c (double nodes):", np.diag(table_double_large))

x,order 0,order 1,order 2
0,1,0,0
1,0.5,-0.5,0
2,0.33333333,-0.16666667,0.16666667


Newton coefficients for 1a: [ 1.         -0.5         0.16666667]


z,order 0,order 1,order 2,order 3,order 4,order 5
0,1,0,0,0,0,0
0,1,-1,0,0,0,0
1,0.5,-0.5,0.5,0,0,0
1,0.5,-0.25,0.25,-0.25,0,0
2,0.33333333,-0.16666667,0.083333333,-0.083333333,0.083333333,0
2,0.33333333,-0.11111111,0.055555556,-0.027777778,0.027777778,-0.027777778


Hermite/Newton coefficients for 1b: [ 1.         -1.          0.5        -0.25        0.08333333 -0.02777778]


x,order 0,order 1,order 2,order 3,order 4,order 5,order 6,order 7,order 8,order 9,order 10
1,0.5,0,0,0,0,0,0,0,0,0,0
1.1,0.4761904762,-0.2380952381,0,0,0,0,0,0,0,0,0
1.2,0.4545454545,-0.2164502165,0.1082251082,0,0,0,0,0,0,0,0
1.3,0.4347826087,-0.1976284585,0.09410878976,-0.04705439488,0,0,0,0,0,0,0
1.4,0.4166666667,-0.1811594203,0.08234519104,-0.03921199573,0.01960599787,0,0,0,0,0,0
1.5,0.4,-0.1666666667,0.07246376812,-0.03293807642,0.01568479829,-0.007842399146,0,0,0,0,0
1.6,0.3846153846,-0.1538461538,0.0641025641,-0.02787068004,0.01266849093,-0.006032614729,0.003016307362,0,0,0,0
1.7,0.3703703704,-0.1424501425,0.05698005698,-0.02374169041,0.01032247409,-0.004692033677,0.002234301753,-0.00111715087,0,0,0
1.8,0.3571428571,-0.1322751323,0.05087505088,-0.02035002035,0.008479175146,-0.00368659789,0.001675726312,-0.0007979649158,0.0003989824428,0,0
1.9,0.3448275862,-0.1231527094,0.04561211458,-0.01754312099,0.007017248397,-0.002923853498,0.001271240653,-0.0005778366562,0.0002751603246,-0.0001375801314,0


Newton coefficients for 1c (simple nodes): [ 5.00000000e-01 -2.38095238e-01  1.08225108e-01 -4.70543949e-02
  1.96059979e-02 -7.84239915e-03  3.01630736e-03 -1.11715087e-03
  3.98982443e-04 -1.37580131e-04  4.58600067e-05]


z,order 0,order 1,order 2,order 3,order 4,order 5,order 6,order 7,order 8,order 9,order 10,order 11,order 12,order 13,order 14,order 15,order 16,order 17,order 18,order 19,order 20,order 21
1,0.5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0.5,-0.25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.1,0.4761904762,-0.2380952381,0.119047619,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.1,0.4761904762,-0.2267573696,0.1133786848,-0.0566893424,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.2,0.4545454545,-0.2164502165,0.1030715316,-0.05153576582,0.02576788291,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.2,0.4545454545,-0.2066115702,0.09838646202,-0.0468506962,0.0234253481,-0.01171267405,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.3,0.4347826087,-0.1976284585,0.0898311175,-0.04277672262,0.02036986791,-0.01018493396,0.005092466969,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.3,0.4347826087,-0.1890359168,0.08592541674,-0.03905700761,0.01859857505,-0.008856464314,0.004428232144,-0.002214116083,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.4,0.4166666667,-0.1811594203,0.07876496534,-0.03580225697,0.01627375317,-0.007749406266,0.003690193496,-0.001845096622,0.0009225486533,0,0,0,0,0,0,0,0,0,0,0,0,0
1.4,0.4166666667,-0.1736111111,0.07548309179,-0.03281873556,0.01491760707,-0.00678073049,0.003228919253,-0.001537580807,0.000768789537,-0.0003843977908,0,0,0,0,0,0,0,0,0,0,0,0


Hermite/Newton coefficients for 1c (double nodes): [ 5.00000000e-01 -2.50000000e-01  1.19047619e-01 -5.66893424e-02
  2.57678829e-02 -1.17126740e-02  5.09246697e-03 -2.21411608e-03
  9.22548653e-04 -3.84397791e-04  1.53768953e-04 -6.15421221e-05
  2.37474862e-05 -9.28068325e-06  3.59177825e-06 -1.18996512e-06
 -7.73817880e-07  5.09381515e-06 -1.38576347e-05  3.50189637e-05
 -6.89431962e-05  1.31690801e-04]


## Exercise 2

For the data

$$
\begin{array}{c|ccccccc}
x & -2 & -1 & 0 & 1 & 2 & 3 & 4 \\
\hline
f(x) & -5 & 1 & 1 & 1 & 7 & 25 & 60
\end{array}
$$

we construct:
- the divided differences table
- the forward finite differences table
- the backward finite differences table

In [3]:
x_data = np.array([-2, -1, 0, 1, 2, 3, 4], dtype=float)
y_data = np.array([-5, 1, 1, 1, 7, 25, 60], dtype=float)

nodes_divided, table_divided = divided_differences_simple(x_data, y_data)
display_difference_table("Exercise 2a - divided differences table", "x", nodes_divided, table_divided)
print("Newton coefficients for 2a:", np.diag(table_divided))

forward_table = forward_differences(y_data)
display_difference_table("Exercise 2b - forward finite differences table", "x", x_data, forward_table)

backward_table = backward_differences(y_data)
display_difference_table("Exercise 2c - backward finite differences table", "x", x_data, backward_table)

x,order 0,order 1,order 2,order 3,order 4,order 5,order 6
-2,-5,0,0,0,0,0,0
-1,1,6,0,0,0,0,0
0,1,0,-3,0,0,0,0
1,1,0,0,1,0,0,0
2,7,6,3,1,0,0,0
3,25,18,6,1,0,0,0
4,60,35,8.5,0.83333333,-0.041666667,-0.0083333333,-0.0013888889


Newton coefficients for 2a: [-5.00000000e+00  6.00000000e+00 -3.00000000e+00  1.00000000e+00
  0.00000000e+00  0.00000000e+00 -1.38888889e-03]


x,order 0,order 1,order 2,order 3,order 4,order 5,order 6
-2,-5,6,-6,6,0,0,-1
-1,1,0,0,6,0,-1,0
0,1,0,6,6,-1,0,0
1,1,6,12,5,0,0,0
2,7,18,17,0,0,0,0
3,25,35,0,0,0,0,0
4,60,0,0,0,0,0,0


x,order 0,order 1,order 2,order 3,order 4,order 5,order 6
-2,-5,0,0,0,0,0,0
-1,1,6,0,0,0,0,0
0,1,0,-6,0,0,0,0
1,1,0,0,6,0,0,0
2,7,6,6,6,0,0,0
3,25,18,12,6,0,0,0
4,60,35,17,5,-1,-1,-1
